# Grokking: запуск оригинального кода `openai/grok` «как есть» на Colab T4

Код статьи Power et al. 2022 (arXiv:2201.02177) **без изменений**, коммит `3d64b1d` (16.03.2024).
Код несовместим с современными `pytorch_lightning` (`self.hparams = hparams` ломается с PL >= 1.3),
поэтому ставим старый стек в отдельный venv на Python 3.9:

| пакет | версия |
|---|---|
| Python | 3.9 |
| torch | 1.10.2+cu113 |
| pytorch_lightning | 1.2.10 |
| torchmetrics | 0.2.0 |
| numpy | 1.23.5 |

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
%%bash
set -e
pip -q install uv
uv venv --python 3.9 /content/venv-grok
[ -d /content/grok ] || git clone -q https://github.com/openai/grok.git /content/grok
cd /content/grok && git checkout -q 3d64b1d8c1d595dd8ebdb7771998823f1b14c7b3 && git log -1 --format='commit %H (%cd)'
uv pip install --python /content/venv-grok/bin/python \
  --extra-index-url https://download.pytorch.org/whl/cu113 --index-strategy unsafe-best-match \
  "torch==1.10.2+cu113" "pytorch_lightning==1.2.10" "torchmetrics==0.2.0" "numpy==1.23.5" \
  "setuptools<60" "pandas<2.1" blobfile tqdm scipy mod matplotlib sympy
uv pip install --python /content/venv-grok/bin/python --no-deps -e /content/grok
/content/venv-grok/bin/python -c "import sys,torch,pytorch_lightning as pl; print(sys.version.split()[0], torch.__version__, pl.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"


In [ ]:
%%writefile /content/plot_metrics.py
"""Plot train/val accuracy from openai/grok CSV logs and report steps-to-threshold.

Usage:
    python tools/plot_metrics.py RUN_DIR [RUN_DIR ...] [--out fig.png] [--thr 99]

RUN_DIR is the logdir passed to scripts/train.py; the script looks for
RUN_DIR/default/version_*/metrics.csv (the latest version).
"""
import argparse
import glob
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


def load(run_dir):
    files = sorted(glob.glob(os.path.join(run_dir, "default", "version_*", "metrics.csv")))
    if not files:
        raise FileNotFoundError(f"no metrics.csv under {run_dir}")
    d = pd.read_csv(files[-1])
    # step logged by Lightning is 0-based; +1 = number of optimizer updates done
    d["updates"] = d["step"] + 1
    train = d.dropna(subset=["train_accuracy"])[["updates", "train_accuracy"]]
    val = d.dropna(subset=["val_accuracy"])[["updates", "val_accuracy", "full_train_acc"]]
    return train, val


def first_reach(df, col, thr):
    hit = df[df[col] >= thr]
    return int(hit["updates"].iloc[0]) if len(hit) else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--out", default="accuracy.png")
    ap.add_argument("--thr", type=float, default=99.0)
    args = ap.parse_args()

    fig, ax = plt.subplots(figsize=(7, 4.2))
    print(f"{'run':40s} {'train>=thr':>11s} {'val>=thr':>10s} {'ratio':>7s} {'final val':>9s}")
    for run in args.runs:
        train, val = load(run)
        name = os.path.basename(os.path.normpath(run))
        # full_train_acc is evaluated on the whole train set at val time (cleaner than
        # the running mean over minibatches in train_accuracy)
        t_hit = first_reach(val, "full_train_acc", args.thr)
        v_hit = first_reach(val, "val_accuracy", args.thr)
        ratio = f"{v_hit / t_hit:.0f}x" if (t_hit and v_hit) else "-"
        print(f"{name:40s} {str(t_hit):>11s} {str(v_hit):>10s} {ratio:>7s} {val['val_accuracy'].iloc[-1]:9.2f}")
        line, = ax.plot(val["updates"], val["full_train_acc"], ls="--", lw=1.2)
        ax.plot(val["updates"], val["val_accuracy"], color=line.get_color(), lw=1.8, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("Optimization steps")
    ax.set_ylabel("Accuracy, %")
    ax.set_ylim(-2, 102)
    ax.grid(alpha=0.3)
    ax.legend(title="solid = val, dashed = train", fontsize=8)
    fig.tight_layout()
    fig.savefig(args.out, dpi=150)
    print("saved", args.out)


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, time, os, shlex
PY = "/content/venv-grok/bin/python"
TRAIN = "/content/grok/scripts/train.py"

def run(name, **kw):
    """Run original scripts/train.py; all flags passed explicitly (code defaults differ from the paper!)."""
    logdir = f"/content/runs/{name}"
    os.makedirs(logdir, exist_ok=True)
    args = " ".join(f"--{k} {shlex.quote(str(v))}" for k, v in kw.items())
    cmd = f"cd {logdir} && {PY} {TRAIN} {args} > train.log 2>&1"
    t0 = time.time()
    rc = subprocess.call(cmd, shell=True)
    dt = time.time() - t0
    print(f"{name}: rc={rc}, {dt/60:.1f} min, {kw.get('max_steps', 100000)/dt:.1f} steps/s")
    if rc != 0:
        print(open(f"{logdir}/train.log").read()[-3000:])
    return logdir

## 1. Smoke-тест (300 шагов) и замер скорости

In [ ]:
run("smoke", math_operator="+", train_data_pct=50, weight_decay=1, max_steps=300, random_seed=0)

## 2. Пилот: конфиг по умолчанию из статьи (AdamW, lr 1e-3, wd 1, β=(0.9,0.98), batch 512, warmup 10)
Ожидание: train ≈100% за ~10³ шагов, val → 99% заметно позже.

In [ ]:
for op, tag in [("+", "add"), ("/", "div")]:
    run(f"pilot_{tag}_wd1_50", math_operator=op, train_data_pct=50, weight_decay=1, max_steps=30000, random_seed=0)

In [ ]:
!{PY} /content/plot_metrics.py /content/runs/pilot_add_wd1_50 /content/runs/pilot_div_wd1_50 --out /content/pilot.png
from IPython.display import Image; Image("/content/pilot.png")

## 3. (Долго) Fig. 1 left как в статье: x/y mod 97, 50%, **Adam без weight decay**, 10⁶ шагов
Запускать только после замера скорости в п.1 — на T4 это, вероятно, несколько часов.
Colab может оборвать сессию — сохраняйте `/content/runs` на Drive.

In [ ]:
# run("fig1_div_nowd_50", math_operator="/", train_data_pct=50, weight_decay=0, max_steps=1000000, random_seed=0)

In [ ]:
# Сохранение результатов на Google Drive
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/grokking && cp -r /content/runs /content/drive/MyDrive/grokking/